# Full PrimeKG：一次训练，四个队列共享

这是**可独立运行**的 notebook，已内置 DistMult 与中断保存代码。云服务器上只需此 `.ipynb`、完整原始 `kg.csv`，以及 Python 环境中的 `torch`、`numpy`、`pandas`、`matplotlib`。

1. 将 notebook 与 `kg.csv` 放在同一个云端工作目录，打开后选择已安装 PyTorch 的 Python kernel。
2. 首次训练保持 `RESUME = False`，从上到下依次运行。
3. 停止时使用 Jupyter 的 **Interrupt Kernel / 中断内核** 按钮；当前优化步骤结束后会保存检查点与嵌入，等待显示 `Saved (interrupted)`。
4. 继续训练时将 `RESUME = True`，重新运行参数单元格和训练单元格；如果重启过 kernel，则从上到下运行。`EPOCHS` 是累计目标轮数。

在本项目中运行时自动使用主目录及共享输出目录。单独上传时默认在当前工作目录读取图谱，输出到 `clinicalKG/shared/FullPrimeKG/`。训练结束后，将共享目录中的结果文件一起下载回本机同名目录。

四个队列共用本 notebook 生成的嵌入及索引；每个队列的患者映射和诊断训练仍分别运行。


## 1. 参数与相对路径

一般只需确认图谱路径和训练参数。首次运行 `RESUME=False`，恢复时改为 `True`。


In [ ]:
from pathlib import Path
import os

# None：自动识别项目主目录；单独上传时使用当前工作目录。
# 也可以填写相对于当前工作目录的路径，例如 ".." 或 "my_project"。
MAIN_FOLDER = None
RAW_KG_FILE = "kg.csv"       # 相对于主目录，也可以填写完整路径。
RESUME = False               # 首次 False；中断后继续训练改成 True。
EPOCHS = 50                  # 累计目标 epoch，不是新增 epoch 数。
BATCH_SIZE = 512
EMBED_DIM = 128
LEARNING_RATE = 0.005
SEED = 2026
SAVE_EVERY_STEPS = 1000      # 同时在每个 epoch 结束和正常中断时保存。
LOG_EVERY_STEPS = 100        # 每隔多少步打印当前 batch loss 和本轮平均 loss。

_cwd = Path.cwd().resolve()
if MAIN_FOLDER is not None:
    PROJECT_ROOT = Path(MAIN_FOLDER).expanduser().resolve()
else:
    _package = next((p for start in (_cwd / "clinicalKG", _cwd)
                     for p in (start, *start.parents)
                     if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file()), None)
    PROJECT_ROOT = _package.parent if _package is not None else _cwd

RAW_KG_PATH = Path(RAW_KG_FILE).expanduser()
if not RAW_KG_PATH.is_absolute():
    RAW_KG_PATH = PROJECT_ROOT / RAW_KG_PATH
# 在原本地项目中复用已有的完整 kg.csv；云端优先读取上面指定的文件。
if RAW_KG_FILE == "kg.csv" and not RAW_KG_PATH.is_file():
    _candidates = [PROJECT_ROOT / "clinicalKG" / "shared" / "FullPrimeKG" / "kg.csv"]
    _candidates += [PROJECT_ROOT / d / kind / "kg.csv"
                    for d in ("ADNI", "AIBL", "NACC", "PPMI")
                    for kind in ("PrimeKG", "DementiaHKG")]
    RAW_KG_PATH = next((p for p in _candidates if p.is_file()), RAW_KG_PATH)
OUTPUT_DIR = PROJECT_ROOT / "clinicalKG" / "shared" / "FullPrimeKG"
SHARED_PRIMEKG = OUTPUT_DIR

def raw_primekg_path():
    return RAW_KG_PATH

print("主目录：", PROJECT_ROOT)
print("完整 PrimeKG：", RAW_KG_PATH)
print("共享输出目录：", OUTPUT_DIR)
print("恢复训练：", RESUME)


## 2. 检查依赖、图谱表头和训练设备

这一单元格只读取表头和少量示例，不开始训练。


In [ ]:
import hashlib
import json
import random
import signal
import threading

import torch
from torch import nn
import numpy as np
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if not RAW_KG_PATH.is_file():
    raise FileNotFoundError(f"请上传完整 kg.csv，或修改 RAW_KG_FILE：{RAW_KG_PATH}")
columns = set(pd.read_csv(RAW_KG_PATH, nrows=0).columns)
if not ({"x_name", "relation", "y_name"} <= columns
        or {"head", "relation", "tail"} <= columns):
    raise ValueError("图谱需包含 x_name/relation/y_name 或 head/relation/tail")
print("PyTorch：", torch.__version__)
print("训练设备：", DEVICE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
print("图谱大小：", round(RAW_KG_PATH.stat().st_size / 1024**3, 3), "GiB")
print("检查点已存在：", (OUTPUT_DIR / "PrimeKG-DistMult-last.pt").is_file())
display(pd.read_csv(RAW_KG_PATH, nrows=3))


## 3. DistMult 与中断保存代码

运行本单元格定义训练函数。这里内置了与 `full_primekg.py` 相同的模型、目标函数和保存逻辑，无需另外上传 `.py`。

使用完整输入图谱，保留全部边记录，不执行 seed 提取、黑名单过滤或 Top-k。沿用原项目的实体名称索引方式。检查点保存模型、优化器、当前 epoch、批次顺序、批次位置和随机状态；恢复时校验输入图谱及超参数。


In [ ]:
class DistMult(nn.Module):
    """Original score, Xavier initialization, and margin-ranking objective."""
    def __init__(self, num_entities, num_relations, embed_dim):
        super().__init__()
        self.num_entities = num_entities
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        self.rel_emb = nn.Embedding(num_relations, embed_dim)
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        self.criterion = nn.MarginRankingLoss(margin=1.0)

    def forward(self, h, r, t):
        return torch.sum(self.ent_emb(h) * self.rel_emb(r) * self.ent_emb(t), dim=1)

    def calculate_loss(self, h, r, t):
        neg_t = torch.randint(0, self.num_entities, (h.size(0),), device=h.device)
        return self.criterion(self(h, r, t), self(h, r, neg_t),
                              torch.ones(h.size(0), device=h.device))

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _atomic_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def load_graph(path):
    """Keep every original edge record, using the original name-based indexing."""
    path = Path(path)
    columns = pd.read_csv(path, nrows=0).columns
    if {"x_name", "relation", "y_name"}.issubset(columns):
        cols = ["x_name", "relation", "y_name"]
    elif {"head", "relation", "tail"}.issubset(columns):
        cols = ["head", "relation", "tail"]
    else:
        raise ValueError("Graph needs x_name/relation/y_name or head/relation/tail")
    print(f"Reading complete graph: {path}", flush=True)
    frame = pd.read_csv(path, usecols=cols, dtype=str)[cols]
    if frame.empty or frame.isna().any().any():
        raise ValueError("Graph is empty or contains missing entity/relation names")
    entities = sorted(set(frame[cols[0]]) | set(frame[cols[2]]))
    relations = sorted(set(frame[cols[1]]))
    ent2id = {name: index for index, name in enumerate(entities)}
    rel2id = {name: index for index, name in enumerate(relations)}
    encoded = np.column_stack((frame[cols[0]].map(ent2id).to_numpy(dtype=np.int64),
                               frame[cols[1]].map(rel2id).to_numpy(dtype=np.int64),
                               frame[cols[2]].map(ent2id).to_numpy(dtype=np.int64)))
    del frame
    return torch.from_numpy(encoded), ent2id, rel2id

def train_full_primekg(raw_kg=None, output_dir=None, *, epochs=50, batch_size=512,
                       embed_dim=128, lr=0.005, device=None, seed=2026,
                       resume=False, save_every_steps=1000, log_every_steps=100):
    """SIGINT/SIGTERM request a save after the current optimizer step finishes."""
    if min(epochs, batch_size, embed_dim, save_every_steps, log_every_steps) <= 0 or lr <= 0:
        raise ValueError("Training parameters must be positive")
    raw_kg = Path(raw_kg) if raw_kg is not None else raw_primekg_path()
    output = Path(output_dir) if output_dir is not None else SHARED_PRIMEKG
    checkpoint = output / "PrimeKG-DistMult-last.pt"
    if checkpoint.exists() and not resume:
        raise FileExistsError(f"Checkpoint exists: {checkpoint}. Use --resume or a new --output-dir.")
    if resume and not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    triplets, ent2id, rel2id = load_graph(raw_kg)
    fingerprint = _sha256(raw_kg)
    print(f"Full PrimeKG: {len(triplets)} records, {len(ent2id)} named entities, "
          f"{len(rel2id)} relations", flush=True)
    output.mkdir(parents=True, exist_ok=True)
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = DistMult(len(ent2id), len(rel2id), embed_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # The next unprocessed row in this epoch's saved permutation.
    epoch, next_row, global_step = 0, 0, 0
    permutation, partial_loss, partial_batches, losses = None, 0.0, 0, []
    config = {"embed_dim": embed_dim, "batch_size": batch_size, "lr": lr,
              "seed": seed, "graph_sha256": fingerprint, "records": len(triplets),
              "entities": len(ent2id), "relations": len(rel2id)}
    if resume:
        # Only load checkpoints generated by this script / trusted training jobs.
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
        if state["config"] != config:
            raise ValueError("Checkpoint graph/settings differ; keep the original graph and training settings")
        if state["entity2id"] != ent2id or state["relation2id"] != rel2id:
            raise ValueError("Checkpoint entity/relation indices differ")
        if state["epoch"] > epochs:
            raise ValueError("--epochs is the total target and is below completed epochs")
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        epoch, next_row, global_step = state["epoch"], state["next_row"], state["global_step"]
        permutation = state["permutation"]
        partial_loss, partial_batches = state["partial_loss"], state["partial_batches"]
        losses = state["loss_history"]
        random.setstate(state["python_rng_state"])
        np.random.set_state(state["numpy_rng_state"])
        torch.set_rng_state(state["torch_rng_state"])
        if device.type == "cuda" and state["cuda_rng_state"] is not None:
            torch.cuda.set_rng_state_all(state["cuda_rng_state"])
        print(f"Resuming epoch {epoch + 1}, next row {next_row}, step {global_step}", flush=True)

    requested_stop = False

    def request_stop(signum, frame):
        nonlocal requested_stop
        requested_stop = True

    def save(reason):
        # CUDA operations complete before serialization; checkpoint is atomically replaced.
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        state = {"format_version": 1, "config": config, "epoch": epoch,
                 "next_row": next_row, "global_step": global_step,
                 "permutation": permutation, "partial_loss": partial_loss,
                 "partial_batches": partial_batches, "loss_history": losses,
                 "model_state_dict": model.state_dict(),
                 "optimizer_state_dict": optimizer.state_dict(),
                 "entity2id": ent2id, "relation2id": rel2id,
                 "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
                 "torch_rng_state": torch.get_rng_state(),
                 "cuda_rng_state": torch.cuda.get_rng_state_all() if device.type == "cuda" else None}
        tmp = checkpoint.with_name(checkpoint.name + ".tmp")
        torch.save(state, tmp)
        os.replace(tmp, checkpoint)
        _atomic_json(output / "PrimeKG-Entity2ID.json", ent2id)
        _atomic_json(output / "PrimeKG-Relation2ID.json", rel2id)
        _atomic_json(output / "PrimeKG-LossHistory.json", losses)
        embedding_path = output / "PrimeKG-Embeddings.npy"
        with embedding_path.with_name(embedding_path.name + ".tmp").open("wb") as handle:
            np.save(handle, model.ent_emb.weight.detach().cpu().numpy())
        os.replace(embedding_path.with_name(embedding_path.name + ".tmp"), embedding_path)
        _atomic_json(output / "PrimeKG-TrainingState.json",
                     {"reason": reason, "completed_epochs": epoch, "next_row": next_row,
                      "global_step": global_step, "partial_epoch_batches": partial_batches,
                      "raw_kg": str(raw_kg.resolve()), **config})
        print(f"Saved ({reason}): {checkpoint}; completed epochs={epoch}, step={global_step}", flush=True)

    previous_handlers = {}
    if threading.current_thread() is threading.main_thread():
        for name in ("SIGINT", "SIGTERM", "SIGBREAK"):
            sig = getattr(signal, name, None)
            if sig is not None:
                previous_handlers[sig] = signal.signal(sig, request_stop)
    try:
        save("start" if not resume else "resume")
        while epoch < epochs and not requested_stop:
            if permutation is None:
                permutation = torch.randperm(len(triplets))
            model.train()
            while next_row < len(triplets) and not requested_stop:
                end = min(next_row + batch_size, len(triplets))
                batch = triplets[permutation[next_row:end]].to(device)
                optimizer.zero_grad()
                loss = model.calculate_loss(batch[:, 0], batch[:, 1], batch[:, 2])
                loss.backward()
                optimizer.step()
                batch_loss = float(loss.detach())
                partial_loss += batch_loss
                partial_batches += 1
                global_step += 1
                next_row = end
                if (partial_batches == 1 or global_step % log_every_steps == 0
                        or next_row == len(triplets) or requested_stop):
                    total_batches = (len(triplets) + batch_size - 1) // batch_size
                    print(f"Epoch [{epoch + 1:03d}/{epochs}] | "
                          f"Batch [{partial_batches}/{total_batches}] | Step {global_step} | "
                          f"Loss: {batch_loss:.4f} | "
                          f"Epoch Mean Loss: {partial_loss / partial_batches:.4f}", flush=True)
                if global_step % save_every_steps == 0:
                    save("periodic")
            if next_row == len(triplets):
                losses.append(partial_loss / partial_batches)
                epoch += 1
                next_row, permutation, partial_loss, partial_batches = 0, None, 0.0, 0
                print(f"Epoch [{epoch:03d}/{epochs}] | Train Loss: {losses[-1]:.4f}", flush=True)
                save("epoch")
        save("interrupted" if requested_stop else "complete")
    finally:
        for sig, handler in previous_handlers.items():
            signal.signal(sig, handler)
    return {"checkpoint": checkpoint, "interrupted": requested_stop,
            "completed_epochs": epoch, "global_step": global_step}

def plot_loss(output_dir=None):
    import matplotlib.pyplot as plt
    output = Path(output_dir) if output_dir is not None else SHARED_PRIMEKG
    losses = json.loads((output / "PrimeKG-LossHistory.json").read_text(encoding="utf-8"))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, len(losses) + 1), losses, label="DistMult Train Loss")
    ax.set(xlabel="Epochs", ylabel="Margin Ranking Loss", title="Full PrimeKG embedding loss")
    ax.grid(True, alpha=0.4)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output / "PrimeKG-LossCurve.png", dpi=300)
    plt.close(fig)


## 4. 开始训练 / 恢复训练

运行后会先读取完整图谱并计算 SHA-256，再进入训练。默认每 100 步打印一次当前 batch loss、本轮平均 loss 和批次进度；第一批、最后一批和中断时也输出。已有检查点时请将第一部分的 `RESUME` 改为 `True`，保持 batch size、维度、学习率和随机种子一致。恢复时可提高累计目标 `EPOCHS`；日志频率 `LOG_EVERY_STEPS` 可以单独调整。

中断使用工具栏的中断按钮；直接关闭或强制重启 kernel 时，下一次从最近已保存的检查点恢复。


In [ ]:
training_result = train_full_primekg(
    raw_kg=RAW_KG_PATH,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    embed_dim=EMBED_DIM,
    lr=LEARNING_RATE,
    device=DEVICE,
    seed=SEED,
    resume=RESUME,
    save_every_steps=SAVE_EVERY_STEPS,
    log_every_steps=LOG_EVERY_STEPS,
)
plot_loss(OUTPUT_DIR)
print("已中断并保存，可续训。" if training_result["interrupted"] else "训练完成。")
print("已完成 epoch：", training_result["completed_epochs"])
print("训练步数：", training_result["global_step"])
print("保存位置：", OUTPUT_DIR)


## 5. 查看保存结果

四队列共享 `PrimeKG-Embeddings.npy`、`PrimeKG-Entity2ID.json`、`PrimeKG-Relation2ID.json`。续训同时保留 `PrimeKG-DistMult-last.pt`。本单元格只查看现有文件，也可在中断后单独运行。


In [ ]:
from IPython.display import Image, display

for path in sorted(OUTPUT_DIR.glob("PrimeKG-*")):
    if path.is_file() and not path.name.endswith(".tmp"):
        print(f"{path.name:36s} {path.stat().st_size / 1024**2:10.2f} MiB")
state_path = OUTPUT_DIR / "PrimeKG-TrainingState.json"
if state_path.is_file():
    saved_progress = json.loads(state_path.read_text(encoding="utf-8"))
    print("保存原因：", saved_progress["reason"])
    print("已完成 epoch：", saved_progress["completed_epochs"])
    print("当前 epoch 下一条记录位置：", saved_progress["next_row"])
embedding_path = OUTPUT_DIR / "PrimeKG-Embeddings.npy"
if embedding_path.is_file():
    embedding = np.load(embedding_path, mmap_mode="r")
    print("共享嵌入 shape：", embedding.shape)
    del embedding
curve_path = OUTPUT_DIR / "PrimeKG-LossCurve.png"
if curve_path.is_file():
    display(Image(filename=str(curve_path)))
